In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Load the text data
print("Loading dataset...")
df = pd.read_csv("../data/cleaned_arxiv_papers.csv") 

# 2. Load embeddings (Takes 1 second instead of 26 minutes!)
print("Loading embeddings...")
embeddings = np.load("../data/arxiv_embeddings.npy")

# 3. Load the model (We need this to convert the user's search query into a vector)
print("Loading model...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("System Ready! Embeddings shape:", embeddings.shape)

Loading dataset...
Loading embeddings...
Loading model...
System Ready! Embeddings shape: (49986, 384)


In [2]:
print(embeddings.dtype)
embeddings

float32


array([[-0.08374164, -0.05528527,  0.01438524, ..., -0.01774349,
        -0.03421708,  0.0255149 ],
       [-0.11149156, -0.05589235, -0.0038947 , ...,  0.02496754,
         0.02245971,  0.0462792 ],
       [ 0.01970266, -0.02694538, -0.01162458, ..., -0.09477925,
        -0.06402169, -0.02941753],
       ...,
       [-0.06649251,  0.07415137, -0.01473149, ...,  0.07772524,
         0.06608181,  0.01619919],
       [-0.0189241 , -0.03860058, -0.00625685, ...,  0.05719413,
        -0.11033951, -0.00414134],
       [-0.1388475 , -0.02017837,  0.02775351, ...,  0.03841157,
        -0.02244471, -0.01448591]], shape=(49986, 384), dtype=float32)



### **1. Why is the dtype `float32`? (The Decimal Point)**

When you look at `embedding.dtype`, it returns `float32`. This means every single number inside your massive array is a 32-bit floating-point number (a decimal number like `0.04561` or `-0.89213`).

* **Why decimals and not whole numbers?** Neural networks capture *subtle* semantic meanings. If they used integers (1, 2, 3), they couldn't capture fine details. A paper might be `0.78` similar to "AI" and `0.21` similar to "Biology". You need decimals to plot these exact coordinates in space.
* **Why `32-bit`?** In Deep Learning, `float32` is the golden industry standard. It gives the model plenty of mathematical precision to represent complex thoughts, but it takes up exactly half the computer memory as a `float64`. This is what allows your computer to calculate the Cosine Similarity so incredibly fast.

### **2. Why are there exactly 384 columns?**

Look at the shape: `(15000, 384)`.

* **15,000** is your number of rows (you successfully encoded 15,000 research papers).
* **384** is your number of columns. As we discussed in **Section 3 of your `dl_nlp_basics.md` notes**, this is the **dimensionality** of your embedding.
* **The "Why":** The specific neural network you downloaded (`all-MiniLM-L6-v2`) was hard-coded by its creators to output exactly 384 numbers. To the AI, every single one of those 384 columns represents a highly abstract "feature" of the text.
* Column 1 might measure how "academic" the text is.
* Column 2 might measure how strongly it relates to "algorithms".
* Column 3 might track "medical context".



*(Note: We can't actually read the columns like that as humans, but mathematically, that is how the neural network organizes the meaning).*

### **Interview Context:**

If an interviewer asks: *"Explain the output shape and type of your embedding matrix."*

**Your Answer:** *"My embedding matrix has a shape of 15,000 by 384, meaning I processed 15,000 documents, and the `MiniLM` model compressed the semantic meaning of each document into a 384-dimensional dense vector. The data type is `float32`, which is standard for neural network weights and embeddings because it perfectly balances computational speed, memory efficiency, and mathematical precision for downstream tasks like Cosine Similarity."*

In [3]:
import faiss

# 1. We must make a copy of the embeddings first because FAISS normalizes "in-place"
# (Meaning it permanently alters the original variable)
faiss_embeddings = embeddings.copy()

# 2. Apply L2 Normalization
faiss.normalize_L2(faiss_embeddings)

print("Embeddings normalized! Ready for the FAISS Index.")

Embeddings normalized! Ready for the FAISS Index.



### **FAISS L2 Normalization & Cosine Similarity**

**The Objective**
To normalize your 384-dimensional document embeddings (forcing their vector lengths to exactly 1) so that FAISS can use its lightning-fast Inner Product engine to calculate mathematically perfect Cosine Similarity scores.

**The Explanation**

* **Why are we normalizing?** In semantic search, the "meaning" of a sentence is stored in the *direction* the vector points, not how long the vector is. By scaling every single vector down to an exact length of 1, we remove any bias caused by vector size and force the math to look *only* at the angle.
* **Why specifically "L2"? (The Math Hack):** FAISS is incredibly fast but **does not** have a built-in Cosine Similarity function. However, it does have a blazing-fast Inner Product (Dot Product) function. The mathematical rule is: `L2 Normalization + Inner Product = Cosine Similarity`. By running `faiss.normalize_L2`, you are hacking FAISS to output the exact Cosine Similarity scores you want while keeping its extreme C++ speed.
* **The Output Range:** Because of this math hack, the scores FAISS will eventually return to you will fall into the standard Cosine Similarity range of **-1.0 to 1.0**:
* **1.0 (Maximum Score):** The vectors point in the exact same direction (identical semantic meaning).
* **0.0 (Neutral Score):** The vectors are perpendicular (unrelated meaning).
* **-1.0 (Minimum Score):** The vectors point in exact opposite directions.



*(Note: In real-world text data, you will almost never see negative scores. Most of your search results will score between `0.0` and `1.0`)*



In [4]:
# index = faiss.IndexFlatIP(384)
# index.add(faiss_embeddings)
# index

In [5]:
import os

# Define the exact path to your data folder
index_path = "../data/paper_faiss.index"

if os.path.exists(index_path):
    print("Loading existing FAISS index")
    index = faiss.read_index(index_path)
else:
    print("Creating new FAISS index")
    faiss.normalize_L2(faiss_embeddings)
    index = faiss.IndexFlatIP(384)
    index.add(faiss_embeddings)
    
    # Save it using the new path variable
    faiss.write_index(index, index_path)
    print("FAISS index saved successfully to the data folder!")

Creating new FAISS index
FAISS index saved successfully to the data folder!


### **FAISS Index Initialization**

**The Objective**
To create the empty structural container (the database "Index") in your computer's RAM, define the mathematical rules it will use to compare vectors, and tell it exactly what size those vectors will be.


**The Explanation**

* **What is `Index`?** In FAISS, an "Index" is essentially the database object. It is the highly optimized C++ data structure that will hold all your research paper embeddings in memory so they can be searched instantly.
* **What is `Flat`?** This tells FAISS how to search. "Flat" means it stores the vectors in a flat, uncompressed array and performs an **Exact Search** (brute-force). It will not skip any vectors; it will compare your search query to every single paper. *(Since you are only using a subset of about 50,000 papers, a Flat index is still incredibly fast. If you had 15 million papers, we would change this to `IVFFlat` to cluster the data).*
* **What is `IP`?** This stands for **Inner Product**. Because you just ran the L2 Normalization step in the previous cell, setting the index to `IP` is the final step of our "Math Hack." It tells the index to use the lightning-fast dot product multiplier, which will perfectly output Cosine Similarity scores.
* **What is `384`?** This is the dimensionality `(d)` of your vectors. FAISS needs to allocate the exact amount of memory required before it accepts any data. You are telling it: *"Get ready, I am about to hand you arrays that have exactly 384 floating-point numbers in them."*
* **Why is the index currently empty?** Right now, you just bought the bookshelf and assembled it, but you haven't put any books on it yet. You have defined the *rules* and the *size* of the database, but it currently holds `0` vectors.

The very next step is to actually take your `faiss_embeddings` matrix and "add" it to this empty index! Let me know when you are ready for that code.

### **Encoding the User Query**

**The Objective**
To take the raw text that a user types into the search bar and convert it into the exact same 384-dimensional mathematical language that our database of research papers uses.


In [6]:
query = "deep learning for medical image analysis"
query_embedding = model.encode(query)
print(query_embedding.shape)

(384,)



**The Explanation**

* **`query = ...`**: This is simply simulating a user typing a search term into your final application.
* **`model.encode(query)`**: You are passing that raw text through the exact same `SentenceTransformer` (`all-MiniLM-L6-v2`) that you used to encode your 15,000 research papers. It is crucial to use the exact same model so that the query and the papers exist in the exact same mathematical space.
* **The Output `(384,)**`: This confirms the model successfully compressed your search phrase into a single, flat 1D array containing 384 floating-point numbers.

**Important Note for the Next Step:**
Remember the `ValueError` we hit way back when we first used `cosine_similarity`? Because FAISS (just like `sklearn`) expects to compare *matrices* (2D arrays), passing a flat `(384,)` array will cause an error. Before we search the FAISS index, we will need to quickly reshape this query vector into a 2D matrix of shape `(1, 384)`!

In [7]:
query_embedding = model.encode([query])
# query_embedding = model.encode(query).reshape(1,-1)
print(query_embedding.shape)
print(query_embedding)

(1, 384)
[[-3.52526680e-02  5.43267019e-02  5.65811843e-02 -4.97096367e-02
  -3.52174528e-02 -2.67348811e-02 -8.23491812e-03  1.52610447e-02
  -1.31554995e-02 -5.45782186e-02 -5.26165217e-02 -1.20118242e-02
  -8.10011178e-02  9.73842740e-02 -1.15813106e-01 -1.68416556e-02
  -9.74306762e-02 -5.52446954e-03 -1.06863745e-01  7.64731225e-03
  -4.89180125e-02 -6.45616604e-03 -3.65177132e-02  2.23113857e-02
   5.65222204e-02  4.34990693e-03  6.29341602e-02 -1.09062947e-01
   3.48555110e-02 -1.03745852e-02  7.24759400e-02 -3.63332741e-02
   1.78154167e-02  1.52814882e-02  4.68112044e-02  7.27785975e-02
  -1.72747020e-02  5.71634509e-02  6.73017837e-03  2.91875172e-02
   4.84563373e-02  5.15047163e-02  3.13240029e-02  5.97998463e-02
   8.63463879e-02 -1.21602137e-03  3.83908004e-02 -1.21437116e-02
   5.59613705e-02  9.04097781e-02 -1.46837588e-02  2.19263658e-02
  -7.75724500e-02  8.67380500e-02  3.64489108e-02 -1.94056355e-03
  -1.39826890e-02 -1.21372230e-02 -1.06874995e-01 -9.59566329e-03
 

In [8]:
faiss.normalize_L2(query_embedding)
query_embedding

array([[-3.52526680e-02,  5.43267019e-02,  5.65811843e-02,
        -4.97096367e-02, -3.52174528e-02, -2.67348811e-02,
        -8.23491812e-03,  1.52610447e-02, -1.31554995e-02,
        -5.45782186e-02, -5.26165217e-02, -1.20118242e-02,
        -8.10011178e-02,  9.73842740e-02, -1.15813106e-01,
        -1.68416556e-02, -9.74306762e-02, -5.52446954e-03,
        -1.06863745e-01,  7.64731225e-03, -4.89180125e-02,
        -6.45616604e-03, -3.65177132e-02,  2.23113857e-02,
         5.65222204e-02,  4.34990693e-03,  6.29341602e-02,
        -1.09062947e-01,  3.48555110e-02, -1.03745852e-02,
         7.24759400e-02, -3.63332741e-02,  1.78154167e-02,
         1.52814882e-02,  4.68112044e-02,  7.27785975e-02,
        -1.72747020e-02,  5.71634509e-02,  6.73017837e-03,
         2.91875172e-02,  4.84563373e-02,  5.15047163e-02,
         3.13240029e-02,  5.97998463e-02,  8.63463879e-02,
        -1.21602137e-03,  3.83908004e-02, -1.21437116e-02,
         5.59613705e-02,  9.04097781e-02, -1.46837588e-0

In [9]:
D, I = index.search(query_embedding, 5)

print("Scores (D):", D)
print("Indices (I):", I)

Scores (D): [[0.7254226  0.7215791  0.6926706  0.67988205 0.6692116 ]]
Indices (I): [[14301 15300 27629 43376 42253]]


In [10]:
print(df.iloc[26063]['title'])

RLOC: Terrain-Aware Legged Locomotion using Reinforcement Learning and
  Optimal Control


This is the grand finale of your search engine backend, bro! You just successfully queried your FAISS database.

Here is the exact breakdown for your notes, including exactly what that `5` means!

### **Executing the FAISS Search**

**The Objective**
To search the FAISS index using the normalized user query, instantly finding the mathematical coordinates of the most relevant research papers and returning their similarity scores and row numbers.

**The Code**

```python
# Search the index!
D, I = index.search(query_embedding, 5)

print("Scores (D):", D)
print("Indices (I):", I)

```

**The Explanation**

* **What is the `5`? (Top-K):** In Machine Learning, this is known as `k`, which stands for how many results you want to retrieve. By passing `5`, you are telling FAISS: *"Find the top 5 closest vectors to my query."* If you wanted to show 10 results on a webpage, you would change this to `10`.
* **What is `D`? (Distances / Scores):** Look at your output: `[[0.72542226 0.7170659  0.71233034 0.6807244  0.6798818 ]]`. Because we used our L2 Normalization + Inner Product hack, these are your **Cosine Similarity Scores**!
* The system found a paper with a `~0.72` match score.
* Notice how they are automatically sorted from highest to lowest. FAISS did all the heavy sorting logic for you!


* **What is `I`? (Indices / Row Numbers):** Look at your output: `[[26063 37161 18030 10466 29766]]`. This is the most important part! FAISS is saying:
* *"The absolute best match is the paper sitting at **Row 37161** in your dataset."*
* *"The second best match is at **Row 37161**."*



### **Next Steps (The Final Piece of the Puzzle)**

Right now, you just have a row number . A user doesn't care about row numbers; they want to read the actual title of the research paper!

Your very last step to complete this project is to take that `I` array, go back to your pandas DataFrame (`df`), and print out the title and abstract for row 

now lets make a finction to just returen top 5 papers and we dont neet to write print [df.iloca] everutime

In [53]:
def search_paper(query,k=5):
    query_embedding=model.encode([query])
    faiss.normalize_L2(query_embedding)
    D,I=index.search(query_embedding,k)
    for score,idx in zip(D[0],I[0]):
        print("Similarity Score: ",score)
        print("Title: ",df.iloc[idx]['title'])
        print("Abstract:" ,df.iloc[idx]['abstract'][:100])
        print() # for space between two papers
    return D,I

Because you only passed 1 query, D and I variables are technically 2D arrays that look like this: [[0.68, 0.67...]]. Adding [0] extracts just that inner list of 5 numbers. The zip() function is a brilliant Python tool that lets us loop through the scores list and the indices list at the exact same time.

In [ ]:
print("User Query:" , query,'\n')
D,I=search_paper(query)

User Query: deep learning for medical image analysis 

Similarity Score:  0.7254226
Title:  An overview of deep learning in medical imaging focusing on MRI
Abstract:   What has happened in machine learning lately, and what does it mean for the
future of medical imag

Similarity Score:  0.7215791
Title:  Medical Imaging with Deep Learning: MIDL 2020 -- Short Paper Track
Abstract:   This compendium gathers all the accepted extended abstracts from the Third
International Conferenc

Similarity Score:  0.6926706
Title:  A Systematic Collection of Medical Image Datasets for Deep Learning
Abstract:   The astounding success made by artificial intelligence (AI) in healthcare and
other fields proves 

Similarity Score:  0.67988205
Title:  Deep Learning in Cardiology
Abstract:   The medical field is creating large amount of data that physicians are unable
to decipher and use 

Similarity Score:  0.6692116
Title:  Deep Learning with Permutation-invariant Operator for Multi-instance
  Histopatholog

if we just explicitly set the device to 0, like we had did earlier as mentioned in documentation, so our code may crash if no gpu is found, therefore we will set device to 0 only when there is a gpu else load model on cpu only 

Huggingface pipeline convention: -1 for CPU and 0 for gpu

In [13]:
from transformers import pipeline
import torch
setdevice = 0 if torch.cuda.is_available() else -1
summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",  # 300MB instead of 1.6GB
    device=setdevice
)

lets verify if the distilbart model is running on our gpu or on our cpu only, it should run on gpu as they are more powerful and better at processing large data

In [14]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0)) # print the gpu name to see on which gpu is it really running

True
NVIDIA GeForce RTX 3050 Laptop GPU


output true means : yes the model is running on our gpu , all good

In [15]:
type(summarizer)

transformers.pipelines.text2text_generation.SummarizationPipeline

In [16]:
summary=summarizer(df.iloc[10466]['abstract'],max_length=120,min_length=40)
print(summary)
# print(summary[0].dtype)
print(type(summary[0]))
print(summary[0]["summary_text"])

[{'summary_text': ' A paper addresses the challenges of estimating social influence with three contributions . It formalizes social influence as a causal effect, one which requires inferences about hypothetical interventions . PIF fits probabilistic models to networks and behavior data to infer variables that serve as proxies for social influence .'}]
<class 'dict'>
 A paper addresses the challenges of estimating social influence with three contributions . It formalizes social influence as a causal effect, one which requires inferences about hypothetical interventions . PIF fits probabilistic models to networks and behavior data to infer variables that serve as proxies for social influence .


`.dtype` only works on NumPy arrays and Pandas Series/DataFrames. It has no meaning on a dictionary hence `AttributeError: 'dict' object has no attribute 'dtype'`. You were probably testing what type the output was, but `type(summary[0])` is the right way to check that, not `.dtype`.

### **Generating AI Summaries from Search Results**

**The Objective**
To loop through the top FAISS search results, fetch the original research paper details from your pandas DataFrame, and use the BART AI model to dynamically read and summarize the massive academic abstracts into bite-sized, readable paragraphs.



In [41]:
for score, idx in zip(D[0], I[0]):
    print("Similarity score:", score)
    print("Title:", df.iloc[idx]["title"])
    print("Abstract (Preview):", df.iloc[idx]["abstract"][:500] + "...\n")
    
    # Generate the summary using the AI pipeline
    summary = summarizer(df.iloc[idx]["abstract"], max_length=120, min_length=20)
    
    # print(summary) # We can comment this out later, it shows the raw dictionary
    print("AI SUMMARY:")
    print(summary[0]["summary_text"],'\n')


Similarity score: 0.7254226
Title: An overview of deep learning in medical imaging focusing on MRI
Abstract (Preview):   What has happened in machine learning lately, and what does it mean for the
future of medical image analysis? Machine learning has witnessed a tremendous
amount of attention over the last few years. The current boom started around
2009 when so-called deep artificial neural networks began outperforming other
established models on a number of important benchmarks. Deep neural networks
are now the state-of-the-art machine learning models across a variety of areas,
from image analysis to natural l...



Your max_length is set to 120, but your input_length is only 84. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=42)


AI SUMMARY:
 Deep neural networks are becoming increasingly popular in the world of image analysis . They can be used to help diagnose and diagnose problems in the medical field . 

Similarity score: 0.7215791
Title: Medical Imaging with Deep Learning: MIDL 2020 -- Short Paper Track
Abstract (Preview):   This compendium gathers all the accepted extended abstracts from the Third
International Conference on Medical Imaging with Deep Learning (MIDL 2020),
held in Montreal, Canada, 6-9 July 2020. Note that only accepted extended
abstracts are listed here, the Proceedings of the MIDL 2020 Full Paper Track
are published in the Proceedings of Machine Learning Research (PMLR).
...

AI SUMMARY:
 This compendium gathers all the accepted extended abstracts from the Third International Conference on Medical Imaging with Deep Learning (MIDL 2020), held in Montreal, Canada, 6-9 July 2020 . 

Similarity score: 0.6926706
Title: A Systematic Collection of Medical Image Datasets for Deep Learning
Abstra

Problem: if token size is less than 120 or 10 so we get that warning mssg to remove it
Solution :  we set max length and min length dynamically to handle edge cases 

here we are using len of text.split() to see basically , apprximately how many tokens are there in this document

len(text) counts characters, including spaces and punctuation, which doesn't reflect how much linguistic content the abstract contains. text.split() counts words, giving a much better estimate of the input length for choosing summary parameters. It's still an approximation because transformer models operate on tokens rather than words. For maximum accuracy, we could use the model's tokenizer to count tokens, but for this application, word count was a simple and effective heuristic.


In [40]:
for score, idx in zip(D[0], I[0]):

    abstract_text = df.iloc[idx]["abstract"]
    input_word_count = len(abstract_text.split())
    # here we are using len of text.split() to see basically , apprximately how many tokens are there in this document

    print("Similarity score:", score)
    print("Title:", df.iloc[idx]["title"])
    print("Abstract (Preview):", abstract_text[:1000] + "...\n") # slice abstract to only 1000 words as it can be extremely large
    
    # Set the max_length to be whichever is SMALLER: 120
    dynamic_max = min(120, int(input_word_count))
    dynamic_min = min(20, int(input_word_count)*0.5)
    # Run the summarizer with the smart limits
    summary = summarizer(abstract_text, max_length=dynamic_max, min_length=dynamic_min, do_sample=False)
    
    # print(summary) # We can comment this out later, it shows the raw dictionary
    print("AI SUMMARY:")
    print(summary[0]["summary_text"],'\n')


Similarity score: 0.7254226
Title: An overview of deep learning in medical imaging focusing on MRI
Abstract (Preview):   What has happened in machine learning lately, and what does it mean for the
future of medical image analysis? Machine learning has witnessed a tremendous
amount of attention over the last few years. The current boom started around
2009 when so-called deep artificial neural networks began outperforming other
established models on a number of important benchmarks. Deep neural networks
are now the state-of-the-art machine learning models across a variety of areas,
from image analysis to natural language processing, and widely deployed in
academia and industry. These developments have a huge potential for medical
imaging technology, medical data analysis, medical diagnostics and healthcare
in general, slowly being realized. We provide a short overview of recent
advances and some associated challenges in machine learning applied to medical
image processing and image analy

### **Systematic Code Breakdown**

* **`for score, idx in zip(D[0], I[0]):`**
* `D[0]` holds your top 5 Cosine Similarity scores. `I[0]` holds the top 5 pandas row numbers (indices).
* The `zip()` function acts like a zipper on a jacket—it pairs them up so you can loop through the score and the row number at the exact same time.


* **`df.iloc[idx]["title"]`**
* You are using pandas "integer location" (`iloc`) to jump directly to the winning row number in your dataset, and extracting the text from the `title` column.


* **`df.iloc[idx]["abstract"][:500]`**
* This grabs the abstract text, but the `[:500]` is a Python string slice. It tells Python to only print the first 500 characters. Academic abstracts are massive; if you print the whole thing, it will flood your VS Code terminal!


* **`summary = summarizer(..., max_length=120, min_length=40)`**
* This is where the magic happens. You pass the *entire* abstract into your Hugging Face `summarizer` pipeline.
* **The Guardrails:** By setting `max_length=120` and `min_length=40`, you are forcing the AI to give you a "Goldilocks" summary—not too long (preventing rambling) and not too short (preventing a useless 3-word answer).


* **`print(summary)` vs `print(summary[0]["summary_text"])**`
* If you just print the raw `summary` variable, Hugging Face returns an ugly list containing a dictionary that looks like this: `[{'summary_text': 'This paper explores deep learning...'}]`.
* To get just the clean, human-readable English, you have to drill into it: `[0]` gets inside the list, and `["summary_text"]` gets the value out of the dictionary.

now we were calling our summarize model for each abstract(all top k results) seperately in above for loop code, we can do it in batch of all saving a lot of computation time  

In [47]:
def old_search_and_summarize(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    allabstracts=[]
    dynamic_max=80
    dynamic_min=20
    for score, idx in zip(D[0], I[0]):
        # Calculate how many words are in the abstract
        abstract_text = df.iloc[idx]["abstract"]
        input_word_count = len(abstract_text.split())

        # Set the min_length to be whichever is SMALLER
        dynamic_min = int((input_word_count)*0.1)
        dynamic_max= int((input_word_count)*0.8)
        allabstracts.append([abstract_text,dynamic_max,dynamic_min])
    
    allsummaries =[]
    for abstracts in allabstracts:
        abstract_text=abstracts[0]
        dynamic_max = abstracts[1]
        dynamic_min = abstracts[2]
        #The Hugging Face summarizer pipeline returns a list of dictionaries, even for a single input.
        summary = summarizer(abstract_text, max_length=dynamic_max, min_length=dynamic_min, do_sample=False)
        allsummaries.append(summary[0]) # append only 0 th index as we are getting a list of dictionaries but as we only gave 1 query we only have 1 element in list of dict so we can access that dict by 0th index
     
    results=[]
    for score, idx, summary in zip(D[0], I[0],allsummaries):
        results.append({
            "score": float(score),
            "title": df.iloc[idx]["title"],
            "summary": summary["summary_text"]
        })
    return results

In [50]:
def search_and_summarize(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    allabstracts=[]
    dynamic_max=80
    dynamic_min=20
    for score, idx in zip(D[0], I[0]):
        # Calculate how many words are in the abstract
        abstract_text = df.iloc[idx]["abstract"]
        input_word_count = len(abstract_text.split())

        # Set the min_length to be whichever is SMALLER
        dynamic_min = min(dynamic_min, int(input_word_count))

        # allabstracts.append([abstract_text,dynamic_max,dynamic_min])
        allabstracts.append(abstract_text)
    
    allsummaries = summarizer(allabstracts, max_length=dynamic_max, min_length=dynamic_min,batch_size=k, do_sample=False)
    # for abstracts in allabstracts:
    #     abstract_text=abstracts[0]
    #     dynamic_max = abstracts[1]
    #     dynamic_min = abstracts[2]
    #     summary = summarizer(abstract_text, max_length=dynamic_max, min_length=dynamic_min,batch_size=k, do_sample=False)
    #     allsummaries.append(summary)
     
    results=[]
    for score, idx, summary in zip(D[0], I[0],allsummaries):
        results.append({
            "score": float(score),
            "title": df.iloc[idx]["title"],
            "summary": summary["summary_text"]
        })
    return results

In [48]:
# called summarizer(...) once per retrieved paper, inside a for loop.
ans=old_search_and_summarize("Deep learning in medical science",k=5) 
ans

[{'score': 0.7058637142181396,
  'title': 'Medical Imaging with Deep Learning: MIDL 2020 -- Short Paper Track',
  'summary': ' This compendium gathers all the accepted extended abstracts from the Third International Conference on Medical Imaging with Deep Learning (MIDL 2020), held in Montreal, Canada, 6-9 July 2020 .'},
 {'score': 0.6966895461082458,
  'title': 'Deep Learning in Cardiology',
  'summary': ' The medical field is creating large amount of data that physicians are unable to decipher and use efficiently . Rule-based expert systems are inefficient in solving complicated medical tasks . Deep learning has emerged as a more accurate and effective technology in a wide range of medical problems .'},
 {'score': 0.6525641679763794,
  'title': 'Bioinformatics and Medicine in the Era of Deep Learning',
  'summary': ' Deep learning has become a disruptive advance in machine learning, giving new live to the long-standing connectionist paradigm in artificial intelligence . It has been a

In [51]:
#The batched version calls it once, on a list of all k retrieved abstracts:
ans=search_and_summarize("Deep learning in medical science",k=5)
ans

[{'score': 0.7058637142181396,
  'title': 'Medical Imaging with Deep Learning: MIDL 2020 -- Short Paper Track',
  'summary': ' This compendium gathers all the accepted extended abstracts from the Third International Conference on Medical Imaging with Deep Learning (MIDL 2020), held in Montreal, Canada, 6-9 July 2020 .'},
 {'score': 0.6966895461082458,
  'title': 'Deep Learning in Cardiology',
  'summary': ' The medical field is creating large amount of data that physicians are unable to decipher and use efficiently . Rule-based expert systems are inefficient in solving complicated medical tasks . Deep learning has emerged as a more accurate and effective technology in a wide range of medical problems .'},
 {'score': 0.6525641679763794,
  'title': 'Bioinformatics and Medicine in the Era of Deep Learning',
  'summary': ' Deep learning has become a disruptive advance in machine learning, giving new live to the long-standing connectionist paradigm in artificial intelligence . It has been a

In [21]:
from keybert import KeyBERT

# kw_model=keybert()
kw_model=KeyBERT(model=model) # as by defualt it may take some another model, so we want the same model to create embeeding the earlier one we used in sentence transofmrer we will use same model
type(kw_model)

keybert._model.KeyBERT

The KeyBERT Error (TypeError: 'module' object is not callable)
The Reason:
In Python, capitalization matters immensely! When you typed import keybert, you imported the entire library (the module). But to actually create the model, you need to call the specific Class inside that library, which is spelled with capital letters (KeyBERT). You tried to "call" the whole folder instead of the specific tool inside of it.

The Fix:
You just need to import the class specifically. Change your cell to this:

In [36]:
text=df.iloc[26063]['abstract']
print(text)
keywords=kw_model.extract_keywords(text)
print(keywords)
print(type(keywords))
print(type(keywords[0]))
print(type(keywords[0][0]))


  We present a unified model-based and data-driven approach for quadrupedal
planning and control to achieve dynamic locomotion over uneven terrain. We
utilize on-board proprioceptive and exteroceptive feedback to map sensory
information and desired base velocity commands into footstep plans using a
reinforcement learning (RL) policy. This RL policy is trained in simulation
over a wide range of procedurally generated terrains. When ran online, the
system tracks the generated footstep plans using a model-based motion
controller. We evaluate the robustness of our method over a wide variety of
complex terrains. It exhibits behaviors which prioritize stability over
aggressive locomotion. Additionally, we introduce two ancillary RL policies for
corrective whole-body motion tracking and recovery control. These policies
account for changes in physical parameters and external perturbations. We train
and evaluate our framework on a complex quadrupedal system, ANYmal version B,
and demonstrate tr

soo we see that we didnt got words deep learning together, so we can fix that by ngrams

In [34]:
finalkeyword=kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 3), stop_words=None)
finalkeyword

[('achieve dynamic locomotion', 0.5621),
 ('dynamic locomotion over', 0.5545),
 ('locomotion over', 0.5346),
 ('dynamic locomotion', 0.5341),
 ('quadrupedal planning', 0.5219)]

So we had a bug that i accidently set stopwords to none and got stopwords, so we can set it to english as well as we can add our own words to a list of stopwords that has english(318 words) + our custom words

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

custom_stopwords = list(ENGLISH_STOP_WORDS) + ['me', 'over', 'locomotion', 'aayush'] # now we wont get locomotion in our keywords , just for a demo 
keywords = kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 3), stop_words=custom_stopwords)
keywords

[('quadrupedal planning control', 0.557),
 ('quadrupedal planning', 0.5219),
 ('approach quadrupedal planning', 0.4876),
 ('driven approach quadrupedal', 0.4446),
 ('generated footstep plans', 0.444)]

now we can also use the keyPhraseVectorizer to remove grammticaly incorrect sentences 
reference : https://keyphrase-vectorizers.readthedocs.io/en/latest/KeyphraseVectorizers.html

Benefits:
1.Extract grammatically accurate keyphases based on their part-of-speech tags.
2.No need to specify n-gram ranges.
3.Get document-keyphrase matrices.
4.Multiple language support.
5.User-defined part-of-speech patterns for keyphrase extraction possible

also we handle edge case by setting top_n to dynamic len 

In [67]:
# pip install keyphrase-vectorizers spacy
# python -m spacy download en_core_web_sm

from keyphrase_vectorizers import KeyphraseCountVectorizer
from keybert import KeyBERT
vectorizer = KeyphraseCountVectorizer()
indexOfPaper= 32093
text=df.iloc[indexOfPaper]['paper_text']
print('Paper title:', df.iloc[indexOfPaper]['title'])
# Extract keywords (no ngram_range needed!)
dynamiclen= len(text.split())
keywords = kw_model.extract_keywords(
    text, 
    vectorizer=vectorizer, 
    top_n=min(int(10),dynamiclen), # if for any edge case , 5 keywords cant be generated to we set the length to just the len of current words in text, its combination should be enough to avoid errors
    use_mmr=True,
    diversity=0.5  # 0 = pure relevance, 1 = max diversity
)
keywords

Paper title: Combinatorial Blocking Bandits with Stochastic Delays


[('combinatorial blocking bandits', 0.7068),
 ('multi - armed bandit problem', 0.6279),
 ('reward feasible subset', 0.5095),
 ('unconditional hardness', 0.3822),
 ('regret guarantees', 0.356),
 ('stochastic delays', 0.3195),
 ('feasibility constraints', 0.314),
 ('heuristic', 0.2985),
 ('rounds', 0.2096),
 ('arm', 0.1593)]

diversify with MMR, since near-duplicate phrases ("deep learning" / "deep neural") can still both surface
so we set diveristy to 0.5 The user now gets a quick overview of the paper

Maximal Marginal Relevance
Its goal is: "Don't just pick the most relevant keywords. Also make sure they are different from each other."

So MMR balances two objectives:

Relevance → Is this phrase similar to the document?
Diversity → Is this phrase different from the keywords we've already selected?

| Approach                    | Candidate Generation | Phrase Quality | Notes                                       |
| --------------------------- | -------------------- | -------------- | ------------------------------------------- |
| n-grams + stop_words=None   | High                 | Poor           | Many incomplete or overlapping phrases      |
| n-grams + English stopwords | Medium               | Better         | Fewer noisy candidates                      |
| KeyphraseCountVectorizer    | Low                  | Excellent      | Generates linguistically valid noun phrases |


Now lets make the final function to get summary + keywords

In [30]:
def getrelevant_papers(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    allabstracts=[]
    dynamic_max=60
    dynamic_min=20
    for score, idx in zip(D[0], I[0]):
        abstract_text = df.iloc[idx]["abstract"]
        input_word_count = len(abstract_text.split())
        dynamic_min = min(dynamic_min, int(input_word_count))
        dynamic_max= max(dynamic_max,int(input_word_count))
        allabstracts.append(abstract_text)
    
    #instead of calling summarizer for each k similiar papers, call them at once using batch 
    allsummaries = summarizer(allabstracts, max_length=dynamic_max, min_length=dynamic_min,batch_size=k, do_sample=False)
    
    keywords = kw_model.extract_keywords(
    allabstracts, 
    vectorizer=vectorizer, 
    top_n=10, 
    use_mmr=True,
    diversity=0.5  # 0 = pure relevance, 1 = max diversity
    )

    # --- BUG FIX HERE ---
    # Normalization: If KeyBERT returns a flat list (happens when there is only 1 document),
    # we wrap it in a list so zip() treats it as a list of lists.
    if len(allabstracts) == 1:
        keywords = [keywords]

        
    results=[]
    for score, idx, summary, Current_keyword in zip(D[0], I[0],allsummaries,keywords):
        results.append({
            "score": float(score),
            "title": df.iloc[idx]["title"],
            "Keywords":Current_keyword,
            "summary": summary["summary_text"]
        })
    return results

In [31]:
finalans=getrelevant_papers("Deep learning in medical science",k=5)
finalans

[{'score': 0.7058637142181396,
  'title': 'Medical Imaging with Deep Learning: MIDL 2020 -- Short Paper Track',
  'Keywords': [('deep learning', 0.5046),
   ('medical imaging', 0.4678),
   ('extended abstracts', 0.3599),
   ('international conference', 0.2616),
   ('pmlr', 0.2591),
   ('full paper track', 0.2224),
   ('compendium', 0.1707),
   ('midl', 0.1639),
   ('machine', 0.1521),
   ('july', 0.0954)],
  'summary': ' This compendium gathers all the accepted extended abstracts from the Third International Conference on Medical Imaging with Deep Learning (MIDL 2020), held in Montreal, Canada, 6-9 July 2020 .'},
 {'score': 0.6966895461082458,
  'title': 'Deep Learning in Cardiology',
  'Keywords': [('deep learning', 0.5414),
   ('cardiology', 0.4496),
   ('structured data', 0.2944),
   ('complicated medical tasks', 0.2888),
   ('expert systems', 0.2831),
   ('field', 0.2088),
   ('imaging modalities', 0.1554),
   ('decipher', 0.1246),
   ('rule', 0.1185),
   ('application papers', 0.1

now we will add the functionality of extracting contextual meaning also of those keywords so a person can easily get to know what that document is about 

In [33]:
queries =["What are recent transformer architectures for NLP?",
"Find papers on diffusion models.",
"Summarize reinforcement learning papers for robotics.",
"What keywords appear in graph neural network research?",
"Compare BERT and RoBERTa papers."]

for q in queries :
    ans=getrelevant_papers(q,k=1)
    print("Query: ",q)
    print("Similarity score :", ans[0]['score'],', title:',ans[0]['title'])
    
    print("keywords: ",ans[0]['Keywords'])
    print("Summary of paper:",ans[0]['summary'],'\n')

Query:  What are recent transformer architectures for NLP?
Similarity score : 0.6746048331260681 , title: Transformers: "The End of History" for NLP?
keywords:  [('deep nlp', 0.5378), ('bert', 0.4487), ('neural architectures', 0.3958), ('xlnet models', 0.355), ('general transformer architecture', 0.3103), ('segment labeling', 0.3096), ('expressiveness', 0.2982), ('tasks', 0.2737), ('future additions', 0.2232), ('important theoretical limitations', 0.1032)]
Summary of paper:  Recent advances in neural architecture, such as the Transformer, have revolutionized the field of Natural Language Processing . A rich family of variations of BERT-style models has been proposed, but they all remain limited in their ability to model certain kinds of information . 

Query:  Find papers on diffusion models.
Similarity score : 0.46393445134162903 , title: Unsupervised learning of anomalous diffusion data
keywords:  [('anomalous diffusion data', 0.6844), ('noisy trajectories', 0.4751), ('stochastic nat